# CLI Quickstart

Start with the common bioinformatics case: a one-column list of human genes from an over-expression result, and return a table of enriched GO terms. This example uses 50 interferon/inflammatory genes and a prepared official GOA human/GO snapshot.

In [1]:
import pandas as pd

from notebooks.display_utils import display_table, metric_cards, repo_root

repo = repo_root()
genes = pd.read_csv(repo / "examples/human_interferon_up_genes.txt", sep="	")
metric_cards([("input genes", f"{len(genes):,}")])
display_table(genes, caption="Input gene list", max_rows=15)

gene_id
IFIT1
IFIT2
IFIT3
ISG15
MX1
OAS1
STAT1
IRF7
CXCL10
RSAD2


In [2]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
mkdir -p notebooks/generated
genesets-rs run examples/human-go-enrich.yaml > notebooks/generated/quickstart_go.tsv
echo "Wrote notebooks/generated/quickstart_go.tsv"

Wrote notebooks/generated/quickstart_go.tsv


In [3]:
results = pd.read_csv(repo / "notebooks/generated/quickstart_go.tsv", sep="	")
metric_cards([
    ("enriched GO terms", f"{len(results):,}"),
    ("query genes", f"{int(results['query_size'].iloc[0]):,}"),
    ("best adjusted p", f"{results['p_adjust_bonferroni'].min():.1e}"),
])

cols = [
    "target_id",
    "target_name",
    "overlap",
    "query_size",
    "target_size",
    "p_adjust_bonferroni",
]
display_table(
    results.sort_values("p_adjust_bonferroni"),
    columns=cols,
    caption="Top enriched GO terms",
    max_rows=15,
    formatters={"p_adjust_bonferroni": lambda value: f"{value:.3e}"},
)

target_id,target_name,overlap,query_size,target_size,p_adjust_bonferroni
GO:0009615,response to virus,32,50,368,1.499e-48
GO:0051607,defense response to virus,29,50,226,3.048e-48
GO:0051707,response to other organism,39,50,1237,1.338e-44
GO:0043207,response to external biotic stimulus,39,50,1240,1.471e-44
GO:0009607,response to biotic stimulus,39,50,1278,4.809e-44
GO:0044419,biological process involved in interspecies interaction between organisms,39,50,1389,1.257e-42
GO:0006952,defense response,38,50,1289,6.566e-42
GO:0009605,response to external stimulus,39,50,1693,2.858e-39
GO:0140546,defense response to symbiont,32,50,736,1.101e-38
GO:0098542,defense response to other organism,32,50,744,1.562e-38


Broad GO terms are often expected at the very top. A useful first pass is to also inspect terms below a target-size threshold.

In [4]:
specific = results.loc[results["target_size"] <= 500].sort_values("p_adjust_bonferroni")
display_table(
    specific,
    columns=cols,
    caption="More specific enriched GO terms",
    max_rows=20,
    formatters={"p_adjust_bonferroni": lambda value: f"{value:.3e}"},
)

target_id,target_name,overlap,query_size,target_size,p_adjust_bonferroni
GO:0009615,response to virus,32,50,368,1.499e-48
GO:0051607,defense response to virus,29,50,226,3.048e-48
GO:0045071,negative regulation of viral genome replication,15,50,49,1.391e-28
GO:0048525,negative regulation of viral process,16,50,75,6.673e-28
GO:0045069,regulation of viral genome replication,16,50,78,1.339e-27
GO:0050792,regulation of viral process,17,50,140,2.027e-25
GO:1903900,regulation of viral life cycle,16,50,112,7.204e-25
GO:0140374,antiviral innate immune response,12,50,76,3.070e-18
GO:0034341,response to type II interferon,12,50,101,1.153e-16
GO:0034340,response to type I interferon,12,50,114,5.280e-16
